# FarmGemma Training on Google Colab - FREE T4 GPU

**Runtime**: T4 GPU (16GB VRAM) - Free!

## Steps:
1. **Runtime** → **Change runtime type** → **T4 GPU**
2. Run cells sequentially

In [ ]:
# Install dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes sentencepiece protobuf
!pip install -q torch torchvision torchaudio
!pip install -q huggingface_hub

print("✅ Dependencies installed!")

In [ ]:
# Mount Google Drive to save checkpoints
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR = "/content/drive/MyDrive/farmgemma"
import os
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"✅ Model will save to {MODEL_DIR}")

In [ ]:
# Login to HuggingFace
from huggingface_hub import login

# Get free token from: https://huggingface.co/join
token = input("Enter HuggingFace token: ")
login(token=token)
print("✅ Logged in to HuggingFace!")

In [ ]:
# Load Gemma 3 1B with 4-bit quantization (works on T4)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("📥 Loading Gemma 3 1B (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    quantization_config=quantization_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
print("✅ Model loaded!")

In [ ]:
# Setup LoRA - Only 1% of parameters need training
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Create synthetic agricultural training data
agricultural_data = [
    {"question": "My rice crop has brown spots on leaves. What disease is this?",
     "answer": "This appears to be Rice Blast (Magnaporthe oryzae). Apply Tricyclazole 75% WP @ 0.6 g/L. For prevention, use resistant varieties like IR-64."},
    {"question": "What is the best time to sow wheat in Punjab?",
     "answer": "The optimal time for wheat sowing in Punjab is between October 25 to November 15. Late sowing after November 20 reduces yield by 20-30 kg/day."},
    {"question": "How to control pink bollworm in cotton?",
     "answer": "Use pheromone traps @ 4-5/acre. Apply Quinalphos 25% EC @ 400 ml/acre. Remove and destroy crop residues after harvest. Use Bt cotton varieties."},
    {"question": "My tomato plants are curling leaves. What should I do?",
     "answer": "Leaf curl in tomato is caused by whitefly. Apply Imidacloprid 17.8% SL @ 0.5 ml/L. Remove infected plants. Use yellow sticky traps to monitor whiteflies."},
    {"question": "What is PM-KISAN scheme eligibility?",
     "answer": "All landholding farmer families with cultivable land are eligible. Each registered farmer receives Rs. 6000/year in 3 installments of Rs. 2000 each."},
    {"question": "How to prepare neem cake fertilizer?",
     "answer": "Apply neem cake @ 150-200 kg/acre 2 weeks before sowing. It acts as nematode control and provides slow-release nitrogen. Mix with soil during land preparation."},
    {"question": "When should I apply urea to paddy?",
     "answer": "Apply urea in 3 splits: 50% at basal, 25% at tillering (21 days after transplanting), and 25% at panicle initiation stage. Follow LCC-based management."},
    {"question": "What is the current wheat price in Azadpur Mandi?",
     "answer": "Wheat prices at Azadpur Mandi range from Rs. 2100-2250/quintal depending on quality. Fair average quality wheat trades around Rs. 2150/quintal."},
    {"question": "How to identify nutrient deficiency in crops?",
     "answer": "Nitrogen: Yellowing of older leaves first. Phosphorus: Purple discoloration. Potassium: Brown scorching on leaf margins. Iron: Yellowing between veins (interveinal chlorosis)."},
    {"question": "What is crop rotation benefit?",
     "answer": "Crop rotation breaks pest cycles, improves soil fertility, reduces weed pressure, and increases yield by 10-15%. Rice-wheat and cotton-legume rotations are common."},
]

# Format for training
def format_sample(sample):
    prompt = f"""You are FarmGemma, an AI assistant for Indian farmers.

Question: {sample['question']}

Answer: {sample['answer']}"""
    return {"text": prompt}

from datasets import Dataset
train_dataset = Dataset.from_list(agricultural_data).map(format_sample)
print(f"✅ Created dataset with {len(train_dataset)} samples")

In [ ]:
# Training configuration
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=f"{MODEL_DIR}/outputs",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    save_steps=100,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
)
print("✅ Training configured!")

In [ ]:
# Start training (T4 will handle this in ~30-60 mins for 10 epochs)
print("🚀 Starting training... This will take 30-60 minutes on T4 GPU")
trainer.train()

In [ ]:
# Save the fine-tuned model
model.save_pretrained(f"{MODEL_DIR}/farmgemma-1b-agri")
tokenizer.save_pretrained(f"{MODEL_DIR}/farmgemma-1b-agri")
print(f"✅ Model saved to {MODEL_DIR}/farmgemma-1b-agri")

In [ ]:
# Test the model
test_prompt = """You are FarmGemma, an AI assistant for Indian farmers.

Question: How to control pests in rice crop?

Answer:"""

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("🧑‍🌾 FarmGemma Response:")
print(response.split("Answer:")[-1].strip())

## Google Colab Free Tier Limits

| Resource | Free T4 |
|----------|---------|
| GPU | T4 16GB (~12-24 hrs continuous) |
| RAM | 12.7 GB |
| Disk | 107 GB temporary |
| Session | Disconnects after idle or ~12 hrs |

## Pro Tips for Colab

1. **Keep session alive**: Click anything every 30 mins
2. **Save frequently**: Use Google Drive mounting
3. **Use smaller model first**: Gemma 1B trains faster
4. **Batch size**: Keep at 4-8 for T4 stability